# 🤖 Introduction to Artificial Intelligence
### A Beginner-Friendly Notebook by **Tarun Jaswani**

---

Welcome! This notebook walks you through AI from the very beginning — no prior experience required.

You will build **five working AI systems**, each one teaching a new concept:

| # | Notebook Section | Concept |
|---|------------------|---------|
| 1 | Hello, AI! | What is a prediction? |
| 2 | Linear Regression | Learning from data with gradient descent |
| 3 | Classification | Spam detector from scratch |
| 4 | Neural Network | Build a neural net with NumPy only |
| 5 | LLM API | Talk to a real frontier AI (Claude) |

> **Run each cell top to bottom.** Every cell is self-contained and explained line by line.

---
## Section 1 — Hello, AI! 👋
### Your First Prediction

**What is AI?**  
At its core, AI is a program that finds patterns in data and uses them to make predictions.

Let's start with the simplest possible example:  
Given **hours studied**, predict **exam score**.

In [ ]:
# Our dataset — (hours studied, exam score) pairs
hours_studied = [1, 2, 3, 4, 5, 6, 7, 8]
exam_scores   = [52, 58, 65, 70, 75, 82, 88, 95]

print('Training Data:')
print(f'  {"Hours":>6}  {"Score":>6}')
for h, s in zip(hours_studied, exam_scores):
    print(f'  {h:>6}  {s:>6}')

In [ ]:
# The simplest AI model: a straight line  →  y = m*x + b
# We calculate the best slope (m) and intercept (b) using least squares

n      = len(hours_studied)
sum_x  = sum(hours_studied)
sum_y  = sum(exam_scores)
sum_xy = sum(x * y for x, y in zip(hours_studied, exam_scores))
sum_x2 = sum(x ** 2 for x in hours_studied)

m = (n * sum_xy - sum_x * sum_y) / (n * sum_x2 - sum_x ** 2)
b = (sum_y - m * sum_x) / n

print(f'Model learned:  score = {m:.2f} × hours + {b:.2f}')
print(f'Interpretation: every extra hour adds ~{m:.1f} points')

In [ ]:
# Make predictions on new data the model has never seen
def predict(hours):
    return m * hours + b

print('Predictions:')
for hours in [3, 5, 9, 12]:
    print(f'  Study {hours:>2} hours  →  predicted score: {predict(hours):.1f}')

# Measure accuracy — Mean Absolute Error
errors = [abs(predict(h) - s) for h, s in zip(hours_studied, exam_scores)]
mae    = sum(errors) / len(errors)
print(f'\nAverage error on training data: {mae:.2f} points  (0 = perfect)')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x_line = np.linspace(0, 10, 100)
y_line = m * x_line + b

plt.figure(figsize=(8, 4))
plt.scatter(hours_studied, exam_scores, color='steelblue', s=80, zorder=5, label='Training data')
plt.plot(x_line, y_line, color='tomato', linewidth=2, label=f'Model: score = {m:.2f}×hours + {b:.2f}')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.title('Section 1 — Your First AI Model (Linear Regression)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**✅ Key Takeaway — Section 1**

- We gave the model **(input → output) pairs** — this is called **training data**
- The model found the **best straight line** through the data
- Now it can **predict** scores for hours it has never seen
- This is **supervised learning** in its simplest form

---
## Section 2 — Linear Regression with Gradient Descent 📉

In Section 1, we solved for the best line mathematically.  
Real AI doesn't do that — it **learns by making mistakes and correcting itself**.

This process is called **Gradient Descent** — the engine behind every neural network.

In [ ]:
# Dataset: house size (sq ft) → price ($1,000s)
house_sizes  = [650, 800, 950, 1100, 1300, 1500, 1700, 1900, 2100, 2400]
house_prices = [180, 210, 245, 280,  320,  365,  405,  445,  490,  550]

# Normalize: scale values to [0, 1] so gradient descent converges faster
max_size  = max(house_sizes)
max_price = max(house_prices)

X = [s / max_size  for s in house_sizes]
Y = [p / max_price for p in house_prices]

print('House dataset (normalised):')
for s, x, p, y in zip(house_sizes, X, house_prices, Y):
    print(f'  size={s:>5} → x={x:.3f}   price={p:>4}k → y={y:.3f}')

In [ ]:
import random
random.seed(42)

# Start with a random guess
weight = random.uniform(0, 1)
bias   = random.uniform(0, 1)

LEARNING_RATE = 0.1
EPOCHS        = 300

def mse(X, Y, w, b):
    return sum((w * x + b - y) ** 2 for x, y in zip(X, Y)) / len(X)

loss_history = []
print(f'Starting → weight={weight:.4f}, bias={bias:.4f}, loss={mse(X,Y,weight,bias):.6f}\n')

for epoch in range(EPOCHS):
    n = len(X)
    grad_w = sum(2 * ((weight * x + bias) - y) * x for x, y in zip(X, Y)) / n
    grad_b = sum(2 * ((weight * x + bias) - y)     for x, y in zip(X, Y)) / n

    weight -= LEARNING_RATE * grad_w
    bias   -= LEARNING_RATE * grad_b

    loss = mse(X, Y, weight, bias)
    loss_history.append(loss)

    if (epoch + 1) % 60 == 0:
        print(f'Epoch {epoch+1:>3}:  loss={loss:.6f}  w={weight:.4f}  b={bias:.4f}')

print(f'\nFinal loss: {loss_history[-1]:.6f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: loss curve
ax1.plot(loss_history, color='tomato', linewidth=2)
ax1.set_title('Loss Curve — Model Improving Over Time')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Mean Squared Error')
ax1.grid(alpha=0.3)

# Right: predictions vs actual
predicted_prices = [(weight * (s / max_size) + bias) * max_price for s in house_sizes]
ax2.scatter(house_sizes, house_prices, color='steelblue', s=80, label='Actual', zorder=5)
ax2.plot(house_sizes, predicted_prices, color='tomato', linewidth=2, label='Predicted')
ax2.set_title('Predictions vs Actual House Prices')
ax2.set_xlabel('Size (sq ft)')
ax2.set_ylabel('Price ($1,000s)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**✅ Key Takeaway — Section 2**

- **Gradient descent** iteratively adjusts weights to reduce error
- The **loss curve** shows the model learning — loss drops each epoch
- **Learning rate** controls the step size — too big = overshoots, too small = too slow
- This exact process (just with many more parameters) is how GPT-4 was trained

---
## Section 3 — Classification: Spam Detector 📧

**Regression** predicts a number. **Classification** predicts a category.

We'll build a spam detector using **Naive Bayes** — a simple but powerful algorithm  
that learns which words are most associated with spam vs legitimate email.

In [ ]:
import math
from collections import defaultdict

training_emails = [
    ('win free money now click here',             'spam'),
    ('congratulations you won a prize claim now', 'spam'),
    ('cheap pills buy now discount offer',        'spam'),
    ('free cash prize winner selected',           'spam'),
    ('urgent click now limited time offer free',  'spam'),
    ('get rich quick investment opportunity now', 'spam'),
    ('you are selected winner free gift claim',   'spam'),
    ('discount medicine buy cheap pills online',  'spam'),
    ('meeting tomorrow at ten in the office',     'ham'),
    ('project update attached please review',     'ham'),
    ('can we schedule a call this week',          'ham'),
    ('your invoice is ready for download',        'ham'),
    ('team lunch friday everyone is welcome',     'ham'),
    ('quarterly report attached for review',      'ham'),
    ('follow up on our discussion yesterday',     'ham'),
    ('budget proposal please share feedback',     'ham'),
]

print(f'Training set: {len(training_emails)} emails')
print(f'  Spam: {sum(1 for _, l in training_emails if l=="spam")}')
print(f'  Ham:  {sum(1 for _, l in training_emails if l=="ham")}')

In [ ]:
class NaiveBayes:
    def __init__(self):
        self.class_counts = defaultdict(int)
        self.word_counts  = defaultdict(lambda: defaultdict(int))
        self.vocab        = set()
        self.total        = 0

    def train(self, emails):
        for text, label in emails:
            self.class_counts[label] += 1
            self.total += 1
            for word in text.lower().split():
                self.word_counts[label][word] += 1
                self.vocab.add(word)

    def predict(self, text):
        words = text.lower().split()
        best_label, best_score = None, float('-inf')
        for label in self.class_counts:
            score = math.log(self.class_counts[label] / self.total)
            total_w = sum(self.word_counts[label].values())
            vocab_n = len(self.vocab)
            for word in words:
                count = self.word_counts[label].get(word, 0) + 1  # Laplace smoothing
                score += math.log(count / (total_w + vocab_n))
            if score > best_score:
                best_score, best_label = score, label
        return best_label

clf = NaiveBayes()
clf.train(training_emails)
print(f'Vocabulary size: {len(clf.vocab)} unique words')

# Show top spam words
spam_scores = {}
for word in clf.vocab:
    spam_scores[word] = (clf.word_counts['spam'].get(word, 0) + 1) / \
                        (clf.word_counts['ham'].get(word,  0) + 1)

print('\nTop spam-indicator words:')
for word, score in sorted(spam_scores.items(), key=lambda x: -x[1])[:6]:
    print(f'  "{word}": {score:.1f}x more likely in spam')

In [ ]:
test_emails = [
    ('free prize click now win money',       'spam'),
    ('project meeting rescheduled to friday','ham'),
    ('cheap offer limited time discount',    'spam'),
    ('please review the attached document',  'ham'),
    ('winner selected claim your free gift', 'spam'),
]

print(f'{"Email":<42} {"Actual":<8} {"Predicted":<10} {"OK?"}')
print('-' * 68)

correct = 0
for text, actual in test_emails:
    pred    = clf.predict(text)
    ok      = '✅' if pred == actual else '❌'
    correct += int(pred == actual)
    display  = text[:40] + '..' if len(text) > 42 else text
    print(f'{display:<42} {actual:<8} {pred:<10} {ok}')

print(f'\nAccuracy: {correct}/{len(test_emails)} = {correct/len(test_emails)*100:.0f}%')

In [ ]:
# Visualise top spam vs ham words
top_spam = sorted(spam_scores.items(), key=lambda x: -x[1])[:8]
top_ham  = sorted(spam_scores.items(), key=lambda x:  x[1])[:8]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

words_s = [w for w, _ in top_spam]
vals_s  = [v for _, v in top_spam]
ax1.barh(words_s[::-1], vals_s[::-1], color='tomato')
ax1.set_title('Top Spam-Indicator Words')
ax1.set_xlabel('Spam / Ham ratio')
ax1.grid(axis='x', alpha=0.3)

words_h = [w for w, _ in top_ham]
vals_h  = [1/v for _, v in top_ham]
ax2.barh(words_h[::-1], vals_h[::-1], color='steelblue')
ax2.set_title('Top Ham-Indicator Words')
ax2.set_xlabel('Ham / Spam ratio')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

**✅ Key Takeaway — Section 3**

- **Classification** predicts categories, not numbers
- **Naive Bayes** computes: *how much does each word shift the probability toward spam?*
- **Laplace smoothing** handles words the model has never seen before
- This same principle (word probabilities) is used in production spam filters today

---
## Section 4 — Neural Network from Scratch 🧠

We'll build a **real neural network** using only NumPy (no PyTorch, no TensorFlow).

**The challenge: XOR**  
A straight line **cannot** separate XOR. A neural network with a hidden layer **can**.  
This is why depth matters.

```
Input(2)  →  Hidden(4, ReLU)  →  Output(1, Sigmoid)
```

In [ ]:
import numpy as np
np.random.seed(42)

X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
Y = np.array([[0],[1],[1],[0]],         dtype=float)

print('XOR Dataset:')
print(f'  {"A":>4}  {"B":>4}  {"XOR":>6}')
for row, label in zip(X, Y):
    print(f'  {int(row[0]):>4}  {int(row[1]):>4}  {int(label[0]):>6}')

print('\nWhy XOR is hard for linear models:')
print('  You cannot draw ONE straight line that separates 0s from 1s.')

# Quick visual proof
fig, ax = plt.subplots(figsize=(4, 4))
colors  = ['steelblue' if y == 0 else 'tomato' for y in Y.flatten()]
labels  = ['0 (blue)' if y == 0 else '1 (red)' for y in Y.flatten()]
for i, (row, c) in enumerate(zip(X, colors)):
    ax.scatter(row[0], row[1], color=c, s=300, zorder=5)
    ax.text(row[0] + 0.05, row[1] + 0.05, f'XOR={int(Y[i][0])}', fontsize=11)
ax.set_xlim(-0.3, 1.4)
ax.set_ylim(-0.3, 1.4)
ax.set_title('XOR — cannot be separated by one line')
ax.set_xlabel('Input A')
ax.set_ylabel('Input B')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Activation functions
def relu(z):              return np.maximum(0, z)
def relu_grad(z):         return (z > 0).astype(float)
def sigmoid(z):           return 1 / (1 + np.exp(-z))
def sigmoid_grad(z):      s = sigmoid(z); return s * (1 - s)

# Initialise weights — He initialisation for ReLU
W1 = np.random.randn(2, 4) * np.sqrt(2/2)
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * np.sqrt(2/4)
b2 = np.zeros((1, 1))

total_params = 2*4 + 4 + 4*1 + 1
print('Network architecture:')
print(f'  Input  layer : 2 neurons')
print(f'  Hidden layer : 4 neurons (ReLU activation)')
print(f'  Output layer : 1 neuron  (Sigmoid activation)')
print(f'  Total params : {total_params}')

In [ ]:
LEARNING_RATE = 0.5
EPOCHS        = 5000
loss_history  = []

for epoch in range(EPOCHS):
    # ── Forward pass ──────────────────────────────
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)          # predictions

    # ── Loss (Binary Cross Entropy) ───────────────
    eps  = 1e-8
    loss = -np.mean(Y * np.log(A2 + eps) + (1 - Y) * np.log(1 - A2 + eps))
    loss_history.append(loss)

    # ── Backward pass (Backpropagation) ───────────
    n       = X.shape[0]
    dA2     = -(Y/(A2+eps)) + ((1-Y)/(1-A2+eps))
    delta2  = dA2 * sigmoid_grad(Z2)
    dW2     = A1.T @ delta2 / n
    db2     = delta2.mean(axis=0, keepdims=True)

    delta1  = (delta2 @ W2.T) * relu_grad(Z1)
    dW1     = X.T @ delta1 / n
    db1     = delta1.mean(axis=0, keepdims=True)

    # ── Update weights ────────────────────────────
    W2 -= LEARNING_RATE * dW2
    b2 -= LEARNING_RATE * db2
    W1 -= LEARNING_RATE * dW1
    b1 -= LEARNING_RATE * db1

    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1:>5}: loss = {loss:.6f}')

print(f'\nFinal loss: {loss_history[-1]:.6f}')

In [ ]:
Z1f, A1f, Z2f, preds = X @ W1 + b1, relu(X @ W1 + b1), (relu(X @ W1 + b1)) @ W2 + b2, sigmoid((relu(X @ W1 + b1)) @ W2 + b2)

print(f'{"A":>4}  {"B":>4}  {"Expected":>9}  {"Raw Pred":>10}  {"Rounded":>8}  {"Correct?"}')
print('-' * 56)
all_ok = True
for i in range(4):
    a, b_val  = int(X[i][0]), int(X[i][1])
    exp       = int(Y[i][0])
    raw       = preds[i][0]
    rounded   = round(raw)
    ok        = '✅' if rounded == exp else '❌'
    if rounded != exp: all_ok = False
    print(f'{a:>4}  {b_val:>4}  {exp:>9}  {raw:>10.4f}  {rounded:>8}  {ok}')

print(f'\n{"🎉 XOR SOLVED — Perfect score!" if all_ok else "⚠️ Some errors — try more epochs."}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(loss_history, color='purple', linewidth=2)
ax1.set_title('Neural Network Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Binary Cross Entropy')
ax1.grid(alpha=0.3)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-0.2, 1.2, 200), np.linspace(-0.2, 1.2, 200))
grid   = np.c_[xx.ravel(), yy.ravel()]
A1g    = relu(grid @ W1 + b1)
zz     = sigmoid(A1g @ W2 + b2).reshape(xx.shape)

ax2.contourf(xx, yy, zz, levels=50, cmap='RdBu_r', alpha=0.6)
ax2.contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2)
clrs = ['steelblue' if y == 0 else 'tomato' for y in Y.flatten()]
for i, (row, c) in enumerate(zip(X, clrs)):
    ax2.scatter(row[0], row[1], color=c, s=200, zorder=5, edgecolors='white', linewidths=2)
    ax2.text(row[0]+0.05, row[1]+0.05, f'XOR={int(Y[i][0])}', fontsize=10, fontweight='bold')
ax2.set_title('Decision Boundary Learned by the Network')
ax2.set_xlabel('Input A')
ax2.set_ylabel('Input B')

plt.tight_layout()
plt.show()

**✅ Key Takeaway — Section 4**

- **Forward pass**: data flows through layers, producing a prediction
- **Backpropagation**: error flows backward, computing gradients for each weight
- **ReLU** introduces non-linearity — without it, stacking layers achieves nothing
- The **decision boundary plot** shows the network learned a curved, non-linear boundary
- This exact architecture (just much larger) powers all modern deep learning

---
## Section 5 — Talking to a Real AI via API 🚀

Everything we built so far is the foundation. Now we connect to **Claude** — a frontier language model by Anthropic.

**Setup** (to use the real API):  
```bash
pip install anthropic
export ANTHROPIC_API_KEY="your-key-here"   # get free at console.anthropic.com
```

**No API key?** The cell runs in demo mode and shows you what the output looks like.

In [ ]:
import os, json, urllib.request

API_KEY   = os.environ.get('ANTHROPIC_API_KEY', '')
DEMO_MODE = not API_KEY

if DEMO_MODE:
    print('⚠️  Running in DEMO MODE (no API key set).')
    print('   Set ANTHROPIC_API_KEY to enable real calls.')
else:
    print(f'✅ API key found — real mode active ({API_KEY[:12]}...)')

print('''
How an API call works:

  You send:  { model, system_prompt, messages }
             ──────────────────────────────────►  Anthropic servers
                                                  (Claude processes)
  You get:   { content: [{ type: text, text: ... }] }
             ◄──────────────────────────────────

Key concepts:
  system   → tells Claude who it is and how to behave
  user     → your message
  assistant→ Claude's reply
  tokens   → units of text (roughly 0.75 words each)
''')

In [ ]:
def ask_claude(user_message, system='You are a helpful AI tutor. Be clear and concise.'):
    """Send a message to Claude and return the reply."""

    DEMO_ANSWERS = {
        'gradient': (
            'Gradient descent is an optimisation algorithm. Imagine you are on a hilly landscape '
            'blindfolded and want to reach the lowest valley. At each step, you feel which direction '
            'slopes downward and take a step that way. In AI, the "hill" is the loss function, '
            'the "position" is the model weights, and each step is a weight update. '
            'Repeat thousands of times and you converge on a good solution.'
        ),
        'difference': (
            'AI (Artificial Intelligence) is the broad field of making machines intelligent. '
            'ML (Machine Learning) is a subset of AI where machines learn from data instead of '
            'being explicitly programmed. Deep Learning is a subset of ML using neural networks '
            'with many layers. Think: AI ⊃ ML ⊃ Deep Learning.'
        ),
        'advice': (
            '1. Start with the fast.ai course — practical first, theory second. '
            '2. Learn Python and NumPy deeply before frameworks. '
            '3. Build one project from scratch — kaggle competition or personal dataset. '
            '4. Read papers: Attention Is All You Need, ResNet, GPT. '
            '5. Understand the math: linear algebra, calculus, probability.'
        ),
    }

    # Demo mode — return a canned answer
    if DEMO_MODE:
        for key, answer in DEMO_ANSWERS.items():
            if key in user_message.lower():
                return f'[DEMO] {answer}'
        return '[DEMO] Great question! (Set ANTHROPIC_API_KEY for a real answer.)'

    # Real API call
    payload = {
        'model'     : 'claude-sonnet-4-20250514',
        'max_tokens': 512,
        'system'    : system,
        'messages'  : [{'role': 'user', 'content': user_message}],
    }
    req = urllib.request.Request(
        url     = 'https://api.anthropic.com/v1/messages',
        data    = json.dumps(payload).encode(),
        headers = {'Content-Type': 'application/json',
                   'x-api-key': API_KEY,
                   'anthropic-version': '2023-06-01'},
        method  = 'POST',
    )
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())['content'][0]['text']

print('ask_claude() function ready.')

In [ ]:
# Question 1 — Explain a concept
print('Q: Explain gradient descent like I am 12 years old.\n')
print(ask_claude('Explain gradient descent like I am 12 years old.'))

In [ ]:
# Question 2 — Comparison question
print('Q: What is the difference between AI, ML, and Deep Learning?\n')
print(ask_claude('What is the difference between AI, ML, and Deep Learning?'))

In [ ]:
# Question 3 — Career advice
print('Q: I am a beginner. What is your advice for learning ML in 2026?\n')
print(ask_claude('I am a beginner. What is your advice for learning ML in 2026?'))

In [ ]:
# Multi-turn conversation example (maintains history)
def chat_demo():
    if DEMO_MODE:
        print('[DEMO] Multi-turn chat — set API key to run interactively.\n')
        convo = [
            ('User', 'What is a transformer?'),
            ('Claude', '[DEMO] A transformer is a neural network architecture based entirely on attention mechanisms. '
                       'Unlike RNNs, it processes all tokens in parallel, making it much faster to train. '
                       'Introduced in "Attention Is All You Need" (2017), it now powers GPT, BERT, Claude, and most modern LLMs.'),
            ('User', 'And what is attention in that context?'),
            ('Claude', '[DEMO] Attention lets each token "look at" all other tokens and decide which ones are most relevant. '
                       'Think of reading a sentence: when you process the word "it", you look back to find what "it" refers to. '
                       'Attention does this mathematically — computing a weighted sum of all token representations, '
                       'where higher weights mean "pay more attention to this token".'),
        ]
        for speaker, text in convo:
            print(f'{speaker}: {text}\n')
        return

    print('Starting multi-turn chat (type "done" to stop):\n')
    history = []
    system  = 'You are an expert ML teacher. Give concise, clear answers with examples.'

    while True:
        user_input = input('You: ').strip()
        if not user_input or user_input.lower() == 'done':
            break
        history.append({'role': 'user', 'content': user_input})

        payload = {'model': 'claude-sonnet-4-20250514', 'max_tokens': 512,
                   'system': system, 'messages': history}
        req = urllib.request.Request(
            url='https://api.anthropic.com/v1/messages',
            data=json.dumps(payload).encode(),
            headers={'Content-Type': 'application/json', 'x-api-key': API_KEY,
                     'anthropic-version': '2023-06-01'},
            method='POST',
        )
        with urllib.request.urlopen(req) as resp:
            reply = json.loads(resp.read())['content'][0]['text']
        history.append({'role': 'assistant', 'content': reply})
        print(f'\nClaude: {reply}\n')

chat_demo()

**✅ Key Takeaway — Section 5**

- An **API call** sends a JSON payload and receives a JSON response — no magic
- **System prompts** shape how the model behaves
- **Multi-turn conversations** work by sending the full history each time
- The model you just called is built on millions of neural networks like the one in Section 4

---
## 🎉 You Have Completed Introduction to AI!

| Section | What You Built | Core Concept |
|---------|---------------|--------------|
| 1 | Study-hours predictor | Supervised learning, prediction |
| 2 | House price model | Gradient descent, loss function |
| 3 | Spam detector | Classification, Naive Bayes |
| 4 | XOR neural network | Backpropagation, hidden layers |
| 5 | Claude API chat | LLMs, API, multi-turn context |

---

### What Next?

| Resource | What It Teaches |
|----------|-----------------|
| [fast.ai](https://course.fast.ai) | Practical deep learning, top-down approach |
| [PyTorch tutorials](https://pytorch.org/tutorials) | The industry-standard DL framework |
| [Andrej Karpathy — makemore](https://github.com/karpathy/makemore) | Build GPT from scratch |
| [3Blue1Brown — Neural Networks](https://www.youtube.com/watch?v=aircAruvnKk) | Stunning visual intuition |
| [Kaggle](https://kaggle.com) | Real datasets, competitions, notebooks |

---

**Author:** Tarun Jaswani — AI Researcher · ML Engineer · Cybersecurity Consultant  
[LinkedIn](https://linkedin.com/in/tarunjaswani) · [Medium](https://medium.com/@tarunjaswani)  

*March 2026*